<a href="https://colab.research.google.com/github/nadia2622/UTS_KecerdasanBuatan/blob/main/Tugas_Reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ngimport data

In [ ]:
# ==============================================================
# CELL: Upload file restoran.xlsx ke Google Colab
# Jalankan cell ini SEKALI di awal, lalu tidak perlu lagi
# ==============================================================

from google.colab import files

print("Silakan pilih file restoran.xlsx dari komputer Anda:")
uploaded = files.upload()

# Konfirmasi file berhasil diupload
for nama_file in uploaded.keys():
    print(f"[✓] File '{nama_file}' berhasil diupload!")

Silakan pilih file restoran.xlsx dari komputer Anda:


Saving restoran.xlsx to restoran.xlsx
[✓] File 'restoran.xlsx' berhasil diupload!


1. Import & Konfigurasi
Pada bagian ini, kita menyiapkan semua "bahan dasar" sebelum program berjalan. Ada dua hal yang dilakukan di sini.
Pertama, kita mengimpor openpyxl. Ini adalah satu-satunya library eksternal yang kita pakai, dan fungsinya hanya untuk membuka dan menyimpan file Excel bukan untuk proses fuzzy-nya. Analoginya seperti kita pakai gunting untuk membuka plastik kemasan, bukan untuk memasak makanannya. Proses memasak (fuzzy-nya) tetap kita lakukan sendiri dari nol.
Kedua, kita mendefinisikan dua "pengaturan global" yang akan dipakai oleh cell-cell berikutnya. FILE_INPUT dan FILE_OUTPUT adalah nama file yang dibaca dan ditulis. RESOLUSI = 1000 adalah pengaturan seberapa presisi proses defuzzifikasi nanti semakin besar angkanya, semakin akurat hasilnya, tapi semakin lama komputasinya. Nilai 1000 sudah lebih dari cukup untuk tugas ini.

In [ ]:
# ==============================================================
# TUGAS: Case Based - Reasoning (Fuzzy Logic)
# Sistem Fuzzy untuk memilih 5 Restoran Terbaik di Bandung
#
# Input : restoran.xlsx (id, pelayanan, harga)
# Output: peringkat.xlsx (5 restoran terbaik + skor)
#
# Library yang dipakai:
# - openpyxl : hanya untuk BACA dan TULIS file Excel
#              (bukan untuk proses fuzzy!)
# - Seluruh proses fuzzy dibangun manual dari nol
# ==============================================================

import openpyxl  # untuk membaca & menulis file .xlsx

# ===== Path File =====
FILE_INPUT  = "restoran.xlsx"    # file data restoran
FILE_OUTPUT = "peringkat.xlsx"   # file hasil 5 terbaik

# ===== Resolusi Defuzzifikasi =====
# Jumlah titik sampel untuk integrasi numerik (centroid)
# Semakin besar = semakin presisi, tapi lebih lambat
RESOLUSI = 1000

2. Fungsi Keanggotaan (trimf & trapmf)
Pada bagian ini, kita membangun dua "alat ukur" inti dari seluruh sistem fuzzy, yaitu fungsi keanggotaan. Kedua fungsi ini adalah fondasi — tanpa mereka, tidak ada yang bisa berjalan.
Fungsi pertama adalah trimf (triangular membership function), yaitu fungsi berbentuk segitiga. Cara kerjanya: nilai di bawah titik a atau di atas titik c mendapat derajat 0 (tidak masuk himpunan sama sekali). Nilai di titik puncak b mendapat derajat 1 (anggota penuh). Nilai di antaranya mendapat derajat antara 0 dan 1 secara proporsional. Kita pakai ini untuk himpunan yang ada di "tengah", seperti Pelayanan=Cukup atau Harga=Sedang.
Fungsi kedua adalah trapmf (trapezoidal membership function), yaitu fungsi berbentuk trapesium. Bedanya dengan segitiga, trapesium punya bagian "datar" di puncaknya — artinya ada rentang nilai yang semuanya mendapat derajat 1 penuh. Kita pakai ini untuk himpunan di "tepi" seperti Pelayanan=Buruk (tepi kiri) dan Pelayanan=Baik (tepi kanan), karena masuk akal kalau pelayanan yang nilainya 90 dan 95 sama-sama dianggap "Baik" sepenuhnya.

In [ ]:
# ==============================================================
# CELL 2: FUNGSI KEANGGOTAAN (Membership Functions)
# Mengubah nilai crisp menjadi derajat keanggotaan (0.0 - 1.0)
# ==============================================================

def trimf(x, a, b, c):
    """
    Fungsi keanggotaan SEGITIGA (Triangular).
    Bentuk: naik dari a ke b, lalu turun dari b ke c.

        1.0       /\
                 /  \
        0.0  ___/    \___
             a    b    c

    Parameter:
        x : nilai crisp yang akan dicek
        a : titik kaki kiri (derajat = 0)
        b : titik puncak   (derajat = 1)
        c : titik kaki kanan (derajat = 0)
    """
    if x <= a or x >= c:
        return 0.0                         # di luar range = tidak anggota
    elif a < x <= b:
        return (x - a) / (b - a)          # sisi naik
    else:
        return (c - x) / (c - b)          # sisi turun


def trapmf(x, a, b, c, d):
    """
    Fungsi keanggotaan TRAPESIUM (Trapezoidal).
    Bentuk: naik dari a ke b, datar dari b ke c, turun dari c ke d.

        1.0       /‾‾‾‾\
                 /      \
        0.0  ___/        \___
             a    b    c    d

    Untuk trapesium KIRI  (tepi kiri): a = b (langsung mulai dari 1)
    Untuk trapesium KANAN (tepi kanan): c = d (tetap 1 sampai ujung)

    Parameter:
        x       : nilai crisp
        a, b    : titik kaki & bahu kiri
        c, d    : titik bahu & kaki kanan
    """
    if x <= a or x >= d:
        return 0.0                         # di luar range
    elif a < x < b:
        return (x - a) / (b - a)          # sisi naik
    elif b <= x <= c:
        return 1.0                         # bagian datar (puncak)
    else:
        return (d - x) / (d - c)          # sisi turun

<>:13: SyntaxWarning: invalid escape sequence '\_'
<>:37: SyntaxWarning: invalid escape sequence '\_'
<>:13: SyntaxWarning: invalid escape sequence '\_'
<>:37: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_3380/2401855088.py:13: SyntaxWarning: invalid escape sequence '\_'
  0.0  ___/    \___
/tmp/ipykernel_3380/2401855088.py:37: SyntaxWarning: invalid escape sequence '\_'
  0.0  ___/        \___


3. Fuzzifikasi
Pada bagian ini, kita mulai "berbicara dalam bahasa fuzzy." Fuzzifikasi adalah proses mengubah nilai angka biasa (crisp) menjadi derajat keanggotaan.
Fungsi fuzzifikasi_pelayanan menerima satu angka, misalnya nilai pelayanan = 71, lalu bertanya tiga pertanyaan sekaligus: "Seberapa Buruk nilai ini? Seberapa Cukup? Seberapa Baik?" Jawabannya bukan ya atau tidak, melainkan tiga angka antara 0 dan 1. Misalnya untuk nilai 71, hasilnya mungkin: Buruk=0, Cukup=0.16, Baik=0.55. Artinya nilai 71 itu "sedikit Cukup" tapi lebih dominan "Baik."
Hal yang sama dilakukan fuzzifikasi_harga untuk nilai harga. Misalnya harga 34.107 ditanya: "Seberapa Murah? Seberapa Sedang? Seberapa Mahal?" Dan jawabannya juga tiga angka.
Kenapa harus tiga angka, bukan satu? Karena itulah inti dari logika fuzzy — satu nilai bisa "sebagian masuk" ke beberapa himpunan sekaligus. Ini jauh lebih realistis dari logika biasa yang hanya mengenal ya atau tidak.

In [ ]:
# ==============================================================
# CELL 3: FUZZIFIKASI
# Mengubah nilai input crisp (pelayanan, harga)
# menjadi derajat keanggotaan untuk setiap himpunan fuzzy
# ==============================================================

def fuzzifikasi_pelayanan(nilai):
    """
    Menghitung derajat keanggotaan nilai Pelayanan (1-100)
    ke dalam 3 himpunan fuzzy: Buruk, Cukup, Baik.

    Desain fungsi keanggotaan:
    - Buruk : Trapesium kiri  [0,  0,  25, 45]
    - Cukup : Segitiga        [30, 55, 75]
    - Baik  : Trapesium kanan [60, 80, 100, 100]
    """
    buruk = trapmf(nilai, 0,  0,  25, 45)   # trapesium kiri
    cukup = trimf (nilai, 30, 55, 75)        # segitiga di tengah
    baik  = trapmf(nilai, 60, 80, 100, 100)  # trapesium kanan

    return {
        "buruk": buruk,
        "cukup": cukup,
        "baik" : baik
    }


def fuzzifikasi_harga(nilai):
    """
    Menghitung derajat keanggotaan nilai Harga (25000-55000)
    ke dalam 3 himpunan fuzzy: Murah, Sedang, Mahal.

    Desain fungsi keanggotaan:
    - Murah : Trapesium kiri  [25000, 25000, 30000, 37000]
    - Sedang: Segitiga        [32000, 40000, 48000]
    - Mahal : Trapesium kanan [43000, 50000, 55000, 55000]
    """
    murah  = trapmf(nilai, 25000, 25000, 30000, 37000)
    sedang = trimf (nilai, 32000, 40000, 48000)
    mahal  = trapmf(nilai, 43000, 50000, 55000, 55000)

    return {
        "murah" : murah,
        "sedang": sedang,
        "mahal" : mahal
    }

4. Inferensi
Pada bagian ini, kita menjalankan "otak" dari sistem fuzzy — tempat semua aturan IF-THEN bekerja. Ini adalah proses inferensi dengan metode Mamdani.
Kita punya 9 aturan. Setiap aturan menggunakan operasi AND yang direpresentasikan dengan MIN. Artinya, kalau aturan berbunyi "IF Pelayanan=Baik AND Harga=Murah THEN Layak", maka kekuatan aturan itu (alpha) adalah nilai terkecil antara derajat "Baik" dan derajat "Murah" dari restoran tersebut. Logikanya: sebuah rantai hanya sekuat mata rantai terlemahnya.
Setelah semua 9 aturan dijalankan, hasilnya dikelompokkan berdasarkan kesimpulan yang sama. Aturan-aturan yang sama-sama menuju "Layak" (aturan 1, 2, 4) digabungkan dengan operasi MAX, artinya kita ambil yang paling kuat di antara mereka. Begitu juga untuk "Cukup_Layak" dan "Tidak_Layak."
Hasil akhir dari fungsi ini adalah tiga angka alpha — satu untuk tiap himpunan output — yang mewakili seberapa kuat kesimpulan tersebut berlaku untuk restoran yang sedang dievaluasi.

In [ ]:
# ==============================================================
# CELL 4: INFERENSI (Fuzzy Inference Engine)
# Menerapkan 9 aturan IF-THEN untuk menghasilkan
# "clipped output" dari setiap aturan.
#
# Metode: Mamdani
# - AND  → operasi MIN  (ambil nilai terkecil)
# - Clipping → potong fungsi keanggotaan output
#   di nilai hasil operasi AND
# ==============================================================

def inferensi(mu_pelayanan, mu_harga):
    """
    Menjalankan 9 aturan inferensi Mamdani.

    Input:
        mu_pelayanan : dict hasil fuzzifikasi pelayanan
                       (kunci: 'buruk', 'cukup', 'baik')
        mu_harga     : dict hasil fuzzifikasi harga
                       (kunci: 'murah', 'sedang', 'mahal')

    Output:
        dict berisi alpha (firing strength) tiap aturan,
        dipetakan ke himpunan output ('tidak_layak',
        'cukup_layak', 'layak')
    """
    # Ambil nilai keanggotaan masing-masing
    buruk  = mu_pelayanan["buruk"]
    cukup  = mu_pelayanan["cukup"]
    baik   = mu_pelayanan["baik"]
    murah  = mu_harga["murah"]
    sedang = mu_harga["sedang"]
    mahal  = mu_harga["mahal"]

    # ===== 9 Aturan Inferensi =====
    # AND = MIN: ambil nilai keanggotaan terkecil dari dua kondisi
    # Hasil = alpha (firing strength) = "seberapa kuat aturan ini aktif"

    # Aturan 1: IF Baik   AND Murah  THEN Layak
    r1  = min(baik,  murah)
    # Aturan 2: IF Baik   AND Sedang THEN Layak
    r2  = min(baik,  sedang)
    # Aturan 3: IF Baik   AND Mahal  THEN Cukup_Layak
    r3  = min(baik,  mahal)
    # Aturan 4: IF Cukup  AND Murah  THEN Layak
    r4  = min(cukup, murah)
    # Aturan 5: IF Cukup  AND Sedang THEN Cukup_Layak
    r5  = min(cukup, sedang)
    # Aturan 6: IF Cukup  AND Mahal  THEN Tidak_Layak
    r6  = min(cukup, mahal)
    # Aturan 7: IF Buruk  AND Murah  THEN Cukup_Layak
    r7  = min(buruk, murah)
    # Aturan 8: IF Buruk  AND Sedang THEN Tidak_Layak
    r8  = min(buruk, sedang)
    # Aturan 9: IF Buruk  AND Mahal  THEN Tidak_Layak
    r9  = min(buruk, mahal)

    # Gabungkan alpha yang menuju himpunan output yang sama
    # dengan operasi MAX (OR antar aturan)
    alpha_tidak_layak  = max(r6, r8, r9)   # aturan 6, 8, 9
    alpha_cukup_layak  = max(r3, r5, r7)   # aturan 3, 5, 7
    alpha_layak        = max(r1, r2, r4)   # aturan 1, 2, 4

    return {
        "tidak_layak" : alpha_tidak_layak,
        "cukup_layak" : alpha_cukup_layak,
        "layak"       : alpha_layak
    }

5. Defuzzifikasi
Pada bagian ini, kita mengubah hasil fuzzy tadi kembali menjadi satu angka nyata yang bisa dipakai untuk membandingkan dan meranking restoran. Metode yang kita pakai adalah Centroid (pusat massa area).
Ada dua bagian di cell ini. Pertama adalah fungsi mu_output yang mendefinisikan bentuk fungsi keanggotaan untuk output. Fungsi ini dipanggil ribuan kali oleh proses defuzzifikasi.
Kedua adalah fungsi defuzzifikasi itu sendiri. Cara kerjanya: kita membagi rentang output (0 sampai 1) menjadi 1001 titik sampel. Untuk setiap titik x, kita hitung tinggi "area fuzzy" yang sudah di-clip oleh nilai alpha dari inferensi tadi. Lalu kita cari titik keseimbangan (centroid) dari seluruh area itu menggunakan rumus rata-rata berbobot.
Analoginya seperti ini: bayangkan area fuzzy itu adalah sepotong tanah dengan kontur tidak rata. Centroid adalah titik di mana tanah itu akan seimbang kalau ditaruh di ujung jari. Titik itulah skor akhir restoran — satu angka antara 0 dan 1.

In [ ]:
# ==============================================================
# CELL 5: DEFUZZIFIKASI (Centroid / Center of Area)
# Mengubah hasil inferensi fuzzy kembali menjadi
# satu angka crisp (skor akhir restoran).
#
# Metode Centroid:
#   skor = Σ(x * μ_agregasi(x)) / Σ(μ_agregasi(x))
#
# x disampel dari 0.0 hingga 1.0 sebanyak RESOLUSI titik.
# ==============================================================

def mu_output(x, nama):
    """
    Fungsi keanggotaan untuk OUTPUT (Skor Kelayakan 0-1).

    Desain:
    - Tidak_Layak : Trapesium kiri  [0.00, 0.00, 0.20, 0.45]
    - Cukup_Layak : Segitiga        [0.30, 0.50, 0.70]
    - Layak       : Trapesium kanan [0.55, 0.80, 1.00, 1.00]
    """
    if nama == "tidak_layak":
        return trapmf(x, 0.00, 0.00, 0.20, 0.45)
    elif nama == "cukup_layak":
        return trimf(x,  0.30, 0.50, 0.70)
    elif nama == "layak":
        return trapmf(x, 0.55, 0.80, 1.00, 1.00)
    return 0.0


def defuzzifikasi(alpha_dict):
    """
    Menghitung skor crisp dengan metode CENTROID.

    Langkah:
    1. Bagi rentang output [0, 1] menjadi RESOLUSI titik
    2. Untuk setiap titik x, hitung μ_agregasi(x):
       → ambil MIN antara alpha dan μ_output(x) tiap himpunan
       → lalu MAX dari semua himpunan (agregasi)
    3. Hitung centroid: Σ(x * μ) / Σ(μ)

    Input:
        alpha_dict : dict hasil inferensi
                     (kunci: 'tidak_layak', 'cukup_layak', 'layak')
    Output:
        skor (float 0-1) hasil defuzzifikasi
    """
    pembilang  = 0.0   # Σ(x * μ)
    penyebut   = 0.0   # Σ(μ)

    # Sampel titik-titik x dari 0 hingga 1
    for i in range(RESOLUSI + 1):
        x = i / RESOLUSI  # x = 0.000, 0.001, 0.002, ..., 1.000

        # Hitung nilai fungsi keanggotaan output di titik x
        # lalu "clip" (potong) di nilai alpha masing-masing himpunan
        mu_tl = min(alpha_dict["tidak_layak"], mu_output(x, "tidak_layak"))
        mu_cl = min(alpha_dict["cukup_layak"], mu_output(x, "cukup_layak"))
        mu_l  = min(alpha_dict["layak"],       mu_output(x, "layak"))

        # Agregasi: ambil nilai MAX dari semua himpunan di titik x
        mu_agregasi = max(mu_tl, mu_cl, mu_l)

        # Akumulasi untuk rumus centroid
        pembilang += x * mu_agregasi
        penyebut  += mu_agregasi

    # Hindari pembagian dengan nol (jika semua alpha = 0)
    if penyebut == 0:
        return 0.0

    return pembilang / penyebut

6. Baca Data dari File
Pada bagian ini, kita membaca isi file restoran.xlsx dan mengubahnya menjadi list Python yang siap diolah.
Fungsi baca_data membuka file Excel menggunakan openpyxl, lalu membaca baris demi baris mulai dari baris kedua (karena baris pertama adalah header). Setiap baris diubah menjadi sebuah dictionary dengan tiga kunci: id, pelayanan, dan harga. Semua nilai dikonversi ke tipe yang tepat — ID menjadi integer, pelayanan dan harga menjadi float — supaya tidak ada masalah saat dihitung nanti.
Ada juga pengecekan baris kosong, untuk jaga-jaga kalau ternyata ada baris kosong di bagian bawah file Excel. Ini praktik yang baik agar program tidak crash di tengah jalan.

In [ ]:
# ==============================================================
# CELL 6: BACA DATA DARI FILE
# Membaca restoran.xlsx menggunakan openpyxl
# (bukan library fuzzy — hanya pembaca Excel)
# ==============================================================

def baca_data(filepath):
    """
    Membaca file Excel restoran.xlsx dan mengembalikan
    list of dict berisi data setiap restoran.

    Output:
        list of dict dengan kunci:
        'id', 'pelayanan', 'harga'
    """
    wb   = openpyxl.load_workbook(filepath)  # buka workbook
    ws   = wb.active                          # ambil sheet aktif
    data = []

    # Iterasi mulai baris ke-2 (baris 1 = header)
    for row in ws.iter_rows(min_row=2, values_only=True):
        id_restoran = row[0]   # kolom A: id Pelanggan
        pelayanan   = row[1]   # kolom B: Pelayanan
        harga       = row[2]   # kolom C: harga

        # Lewati baris kosong
        if id_restoran is None:
            continue

        data.append({
            "id"       : int(id_restoran),
            "pelayanan": float(pelayanan),
            "harga"    : float(harga)
        })

    print(f"[INFO] Berhasil membaca {len(data)} data restoran.")
    return data

7. Simpan Output ke File
Pada bagian ini, kita menulis hasil 5 restoran terbaik ke dalam file peringkat.xlsx yang baru.
Fungsi simpan_output membuat workbook Excel baru dari nol, menulis baris header terlebih dahulu, lalu mengisi data 5 restoran terbaik baris per baris. Nomor peringkat ditambahkan secara otomatis menggunakan enumerate(hasil, start=1) — artinya kita menghitung dari 1, bukan dari 0.
Nilai skor dibulatkan ke 6 angka desimal sebelum ditulis, supaya file output terlihat rapi dan tidak menampilkan angka seperti 0.7234567891234567. Di akhir, file disimpan ke disk dengan wb.save().


In [ ]:
# ==============================================================
# CELL 7: SIMPAN OUTPUT KE FILE
# Menulis hasil 5 restoran terbaik ke peringkat.xlsx
# ==============================================================

def simpan_output(filepath, hasil):
    """
    Menyimpan daftar 5 restoran terbaik ke file Excel.

    Input:
        filepath : nama file output (string)
        hasil    : list of dict, sudah diurutkan,
                   berisi 5 restoran terbaik

    Format output (kolom):
        Peringkat | ID Restoran | Pelayanan | Harga | Skor
    """
    wb = openpyxl.Workbook()   # buat workbook baru
    ws = wb.active
    ws.title = "Peringkat Restoran"

    # Tulis header
    header = ["Peringkat", "ID Restoran", "Pelayanan", "Harga", "Skor Kelayakan"]
    ws.append(header)

    # Tulis data 5 terbaik
    for peringkat, resto in enumerate(hasil, start=1):
        ws.append([
            peringkat,
            resto["id"],
            resto["pelayanan"],
            resto["harga"],
            round(resto["skor"]*100, 2)   # bulatkan 6 desimal
        ])

    wb.save(filepath)
    print(f"[INFO] Output berhasil disimpan ke '{filepath}'")

8. Fungsi Utama
Pada bagian ini, semua cell sebelumnya akhirnya dirangkai menjadi satu alur kerja yang utuh. Ini adalah "sutradara" yang mengatur siapa tampil kapan.
Alurnya sangat linear dan mudah diikuti. Pertama, data dibaca dari file. Kedua, program masuk ke dalam sebuah loop yang memproses setiap satu restoran secara berurutan — fuzzifikasi dulu, lalu inferensi, lalu defuzzifikasi. Hasil skor setiap restoran dikumpulkan ke dalam list hasil_semua.
Setelah semua 100 restoran selesai diproses, list itu diurutkan dari skor tertinggi ke terendah menggunakan sort dengan reverse=True. Lima elemen pertama dari list yang sudah terurut itulah yang menjadi pemenang. Terakhir, hasilnya disimpan ke file dan ditampilkan ke layar dalam format tabel yang rapi.

In [ ]:
# ==============================================================
# CELL 8: FUNGSI UTAMA
# Merangkai seluruh proses fuzzy dari awal hingga akhir:
#   Baca Data → Fuzzifikasi → Inferensi →
#   Defuzzifikasi → Urutkan → Simpan
# ==============================================================

def jalankan_sistem_fuzzy():
    """
    Pipeline lengkap sistem fuzzy:
    1. Baca data dari restoran.xlsx
    2. Untuk setiap restoran: fuzzifikasi → inferensi → defuzzifikasi
    3. Urutkan berdasarkan skor (tertinggi = terbaik)
    4. Ambil 5 teratas
    5. Simpan ke peringkat.xlsx
    6. Tampilkan hasilnya
    """
    print("=" * 60)
    print("  SISTEM FUZZY - PEMILIHAN 5 RESTORAN TERBAIK BANDUNG")
    print("=" * 60)

    # ── LANGKAH 1: Baca data ─────────────────────────────────
    data = baca_data(FILE_INPUT)

    # ── LANGKAH 2, 3, 4: Proses fuzzy tiap restoran ─────────
    print("\n[PROSES] Menjalankan fuzzy inference untuk 100 restoran...")
    hasil_semua = []

    for resto in data:
        pel = resto["pelayanan"]
        hrg = resto["harga"]

        # FUZZIFIKASI
        mu_pel = fuzzifikasi_pelayanan(pel)
        mu_hrg = fuzzifikasi_harga(hrg)

        # INFERENSI
        alpha = inferensi(mu_pel, mu_hrg)

        # DEFUZZIFIKASI
        skor = defuzzifikasi(alpha)

        # Simpan hasil
        hasil_semua.append({
            "id"       : resto["id"],
            "pelayanan": pel,
            "harga"    : hrg,
            "skor"     : skor
        })

    # ── LANGKAH 5: Urutkan berdasarkan skor (descending) ────
    # key=lambda r: r["skor"] → urutkan berdasarkan nilai "skor"
    # reverse=True → dari terbesar ke terkecil
    hasil_semua.sort(key=lambda r: r["skor"], reverse=True)

    # ── LANGKAH 6: Ambil 5 teratas ──────────────────────────
    top5 = hasil_semua[:5]

    # ── LANGKAH 7: Simpan ke file ────────────────────────────
    simpan_output(FILE_OUTPUT, top5)

    # ── LANGKAH 8: Tampilkan ke layar ────────────────────────
    print("\n" + "=" * 60)
    print("       5 RESTORAN TERBAIK PILIHAN SISTEM FUZZY")
    print("=" * 60)
    print(f"{'Peringkat':<12}{'ID':>6}{'Pelayanan':>12}{'Harga':>12}{'Skor':>12}")
    print("-" * 60)

    for rank, resto in enumerate(top5, start=1):
        print(
            f"{rank:<12}"
            f"{int(resto['id']):>6}"
            f"{int(resto['pelayanan']):>12}"
            f"{int(resto['harga']):>12}"
            f"{resto['skor']*100:>12.2f}"
        )

    print("=" * 60)
    print(f"\n[INFO] File output tersimpan di: {FILE_OUTPUT}")
    return top5

9. Jalankan Program
Pada bagian ini, hanya ada satu baris yang berarti: memanggil jalankan_sistem_fuzzy().
Semua fungsi di Cell 1–8 hanya mendefinisikan apa yang harus dilakukan — mereka tidak langsung bekerja begitu di-run. Baru ketika Cell 9 dijalankan, seluruh mesin mulai berputar: data dibaca, 100 restoran diproses satu per satu, diurutkan, lalu 5 terbaik ditampilkan dan disimpan. Cell ini adalah tombol "START" dari seluruh program.

In [ ]:
# ==============================================================
# CELL 9: JALANKAN PROGRAM
# Panggil fungsi utama dan tampilkan hasil akhir
# ==============================================================

# Pastikan file restoran.xlsx sudah diupload ke Google Colab!
# Caranya: klik ikon folder di sidebar kiri → Upload

hasil_akhir = jalankan_sistem_fuzzy()

  SISTEM FUZZY - PEMILIHAN 5 RESTORAN TERBAIK BANDUNG
[INFO] Berhasil membaca 100 data restoran.

[PROSES] Menjalankan fuzzy inference untuk 100 restoran...
[INFO] Output berhasil disimpan ke 'peringkat.xlsx'

       5 RESTORAN TERBAIK PILIHAN SISTEM FUZZY
Peringkat       ID   Pelayanan       Harga        Skor
------------------------------------------------------------
1               78          86       27315       82.92
2               86          84       29811       82.92
3               35          87       40607       82.58
4               22          99       39211       82.48
5               81          52       28905       82.38

[INFO] File output tersimpan di: peringkat.xlsx
